# 실습: 제어 파라미터

## 학습 목표

- `temperature`, `top_p`, `top_k`, `max_output_tokens`의 역할을 이해한다
- 각 파라미터가 출력에 미치는 영향을 직접 실습한다
- 용도별 최적 파라미터 조합을 익힌다
- `stop_sequences`를 활용할 수 있다
- API 요청 한도 오류를 이해하고 셀 단위로 안전하게 실행한다

---
## 0. 환경 설정

아래 셀을 먼저 실행해 공통 함수와 클라이언트를 준비합니다.

In [1]:
import os
import re
import time
from dotenv import load_dotenv
from google import genai
from google.genai import errors, types

load_dotenv()

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODEL = "gemini-2.5-flash"

print(f"모델: {MODEL}")
print("클라이언트 준비 완료")

모델: gemini-2.5-flash
클라이언트 준비 완료


In [2]:
# ── 공통 헬퍼 함수 ────────────────────────────────────────────────────

def get_retry_delay_seconds(error: errors.APIError) -> float:
    """API 오류 응답에서 권장 재시도 대기 시간을 가져온다."""
    if isinstance(error.details, dict):
        for detail in error.details.get("error", {}).get("details", []):
            retry_delay = detail.get("retryDelay")
            if retry_delay:
                match = re.fullmatch(r"(\d+(?:\.\d+)?)s", retry_delay)
                if match:
                    return float(match.group(1)) + 1

    if error.message:
        match = re.search(r"retry in (\d+(?:\.\d+)?)s", error.message, re.I)
        if match:
            return float(match.group(1)) + 1

    return 15.0


def get_finish_reason(response: types.GenerateContentResponse) -> str:
    if response.candidates:
        reason = response.candidates[0].finish_reason
        if reason is None:
            return "UNKNOWN"
        return getattr(reason, "name", str(reason))

    prompt_feedback = getattr(response, "prompt_feedback", None)
    block_reason = getattr(prompt_feedback, "block_reason", None)
    if block_reason:
        return f"PROMPT_BLOCKED/{getattr(block_reason, 'name', block_reason)}"

    return "UNKNOWN"


def extract_text(response: types.GenerateContentResponse) -> str:
    text = response.text
    if text is not None:
        return text
    return f"[텍스트 응답 없음: finish_reason={get_finish_reason(response)}]"


def generate_response(prompt: str, max_retries: int = 1, **kwargs) -> types.GenerateContentResponse:
    for attempt in range(max_retries + 1):
        try:
            return client.models.generate_content(
                model=MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(**kwargs),
            )
        except errors.APIError as error:
            can_retry = error.code == 429 and attempt < max_retries
            if not can_retry:
                raise
            delay = get_retry_delay_seconds(error)
            print(f"[quota] 요청 한도 초과. {delay:.1f}초 대기 후 재시도...")
            time.sleep(delay)

    raise RuntimeError("Gemini API 요청 재시도에 실패했습니다.")


def generate(prompt: str, **kwargs) -> str:
    try:
        return extract_text(generate_response(prompt, **kwargs))
    except errors.APIError as error:
        if error.code == 429:
            return "[요청 한도 초과: 1분 정도 기다린 뒤 다시 실행하세요]"
        return f"[API 오류 {error.code}: {error.message}]"


def generate_with_finish(prompt: str, **kwargs) -> tuple[str, str]:
    try:
        response = generate_response(prompt, **kwargs)
    except errors.APIError as error:
        if error.code == 429:
            return "[요청 한도 초과]", "API_ERROR/429"
        return f"[API 오류 {error.code}]", f"API_ERROR/{error.status or error.code}"
    return extract_text(response), get_finish_reason(response)


print("공통 헬퍼 함수 준비 완료")

공통 헬퍼 함수 준비 완료


---
## 1. 파라미터 전체 구조

```python
from google.genai import types

config = types.GenerateContentConfig(
    # 창의성 제어
    temperature=0.7,        # 0.0 ~ 2.0
    top_p=0.95,             # 0.0 ~ 1.0
    top_k=40,               # 정수

    # 길이 제어
    max_output_tokens=1000, # 최대 출력 토큰

    # 조기 종료
    stop_sequences=["---", "끝"],

    # 역할 지시
    system_instruction="당신은 친절한 도우미입니다.",
)
```

---
## 2. temperature — 창의성과 결정성

```
다음 토큰 후보 확률 분포:
"오늘 날씨가 ___"

temperature = 0.0  (결정적)
  맑다: 45% ████████████ ← 항상 선택
  흐리다: 30% ████████
  덥다: 15% ████

temperature = 1.0  (균형)
  맑다: 45%  ← 자주 선택
  흐리다: 30% ← 가끔 선택
  덥다: 15%  ← 드물게 선택

temperature = 2.0  (매우 무작위)
  거의 균등 분포 → 예측 불가능한 출력
```

| 용도 | temperature | 이유 |
|------|-------------|------|
| 사실 기반 Q&A | 0.0 ~ 0.3 | 일관되고 정확한 답변 |
| 코드 생성 | 0.2 ~ 0.5 | 정확하되 약간의 다양성 |
| 일반 대화 | 0.7 ~ 1.0 | 자연스러운 변화 |
| 창작·브레인스토밍 | 1.0 ~ 1.5 | 다양한 아이디어 |
| 시/예술적 글쓰기 | 1.5 ~ 2.0 | 매우 창의적 |

In [3]:
# temperature = 0.0 (결정적)
prompt = "새로운 카페 이름을 하나 제안해주세요. 이름만 답해주세요."

result = generate(prompt, temperature=0.0, max_output_tokens=30)
print(f"temperature=0.0: {result.strip()}")

temperature=0.0: 오아


In [4]:
# temperature = 1.0 (균형)
result = generate(prompt, temperature=1.0, max_output_tokens=30)
print(f"temperature=1.0: {result.strip()}")

temperature=1.0: 오아


In [5]:
# temperature = 2.0 (매우 무작위)
result = generate(prompt, temperature=2.0, max_output_tokens=30)
print(f"temperature=2.0: {result.strip()}")

temperature=2.0: [텍스트 응답 없음: finish_reason=MAX_TOKENS]


---
## 3. top_p — 핵심 후보군 제한 (Nucleus Sampling)

상위 확률의 토큰들을 누적했을 때 `top_p`에 도달하는 후보군에서만 선택합니다.

```
토큰 확률:
  맑다:   45% ─┐
  흐리다: 30% ─┤ top_p=0.8 → 이 3개에서만 선택
  덥다:   15% ─┘  (합계 90% > 80%)
  춥다:    7%  ← 제외
  눈이 온다: 3% ← 제외
```

> `temperature`와 함께 사용할 때는 주로 `top_p=0.95` 고정, `temperature`로 조절합니다.

In [6]:
# top_p = 0.5 (보수적: 높은 확률 토큰만)
prompt = "하늘을 묘사하는 형용사 하나만 답해주세요."

result = generate(prompt, temperature=1.0, top_p=0.5, max_output_tokens=10)
print(f"top_p=0.5: {result.strip()}")

top_p=0.5: [텍스트 응답 없음: finish_reason=MAX_TOKENS]


In [7]:
# top_p = 1.0 (다양: 모든 후보 허용)
result = generate(prompt, temperature=1.0, top_p=1.0, max_output_tokens=10)
print(f"top_p=1.0: {result.strip()}")

top_p=1.0: [텍스트 응답 없음: finish_reason=MAX_TOKENS]


---
## 4. top_k — 후보 수 직접 제한

상위 k개의 토큰만 후보로 허용합니다.

| top_k 값 | 특징 |
|----------|------|
| 1 | Greedy decoding (항상 최고 확률) |
| 10 ~ 40 | 균형 잡힌 기본값 |
| 100+ | 창의적, 예측 불가 |

In [8]:
import time
# top_k 비교
prompt = "오늘 점심 메뉴를 하나 추천해주세요. 메뉴 이름만 답해주세요."

for top_k in [1, 10, 40, 100]:
    result = generate(prompt, temperature=1.0, top_k=top_k, max_output_tokens=20)
    print(f"top_k={top_k:>3}: {result.strip()}")
    time.sleep(2)  # 요청 간 간격을 두어 API 한도 초과 방지

top_k=  1: [텍스트 응답 없음: finish_reason=MAX_TOKENS]
top_k= 10: [텍스트 응답 없음: finish_reason=MAX_TOKENS]
top_k= 40: [텍스트 응답 없음: finish_reason=MAX_TOKENS]
top_k=100: 비


---
## 5. max_output_tokens — 응답 길이 제한

> 토큰 한도 도달 시 `finish_reason = "MAX_TOKENS"` — 응답이 잘릴 수 있음

In [ ]:
# max_output_tokens = 30 (매우 짧게)
prompt = "Python의 주요 특징을 설명해주세요."

text, finish = generate_with_finish(prompt, max_output_tokens=30, temperature=0.3)
print(f"max_output_tokens=30 (finish_reason={finish}):")
print(text)

[quota] 요청 한도 초과. 50.0초 대기 후 재시도...


In [ ]:
# max_output_tokens = 100
text, finish = generate_with_finish(prompt, max_output_tokens=100, temperature=0.3)
print(f"max_output_tokens=100 (finish_reason={finish}):")
print(text)

In [ ]:
# max_output_tokens = 300
text, finish = generate_with_finish(prompt, max_output_tokens=300, temperature=0.3)
print(f"max_output_tokens=300 (finish_reason={finish}):")
print(text)

---
## 6. stop_sequences — 특정 문자열에서 멈추기

```python
config = types.GenerateContentConfig(
    stop_sequences=["STOP", "---", "\n\n"],
)
```

활용 예:
```
프롬프트: "1번 항목을 작성해줘. 끝나면 '---'를 출력해."
응답: "1번은 ... 입니다. ---"  ← '---' 직전에서 자름
```

In [ ]:
# stop_sequences 없이
prompt = (
    "Python 학습 단계를 번호 목록으로 나열해줘. "
    "각 항목 뒤에 마침표를 붙여줘. "
    "5개 작성 후 '---끝---' 을 출력해."
)

text = generate(prompt, temperature=0.3, max_output_tokens=300)
print("[stop_sequences 없음]")
print(text)

In [ ]:
# stop_sequences 있음 — '---끝---' 이전에서 자동 중단
text = generate(
    prompt,
    temperature=0.3,
    max_output_tokens=300,
    stop_sequences=["---끝---"],
)
print("[stop_sequences=['---끝---'] 있음]")
print(text)

---
## 7. finish_reason 해석

```python
response = client.models.generate_content(...)
reason = response.candidates[0].finish_reason

# STOP        — 정상 종료
# MAX_TOKENS  — 토큰 한도 도달 (응답 잘림)
# SAFETY      — 안전 필터 작동
# RECITATION  — 저작권 관련 중단
```

In [ ]:
# finish_reason 직접 확인
response = generate_response(
    "파이썬이란 무엇인지 한 문장으로 설명해주세요.",
    temperature=0.3,
    max_output_tokens=200,
)

print(f"finish_reason: {get_finish_reason(response)}")
print(f"응답: {extract_text(response)}")

---
## 8. 파라미터 조합 레시피

### 용도별 권장 설정

| 용도 | temperature | top_p | top_k | max_output_tokens |
|------|-------------|-------|-------|-------------------|
| 사실 기반 Q&A | 0.1 | 0.8 | 20 | 500 |
| 자연스러운 챗봇 | 0.8 | 0.95 | 40 | 1000 |
| 창의적 글쓰기 | 1.4 | 0.99 | 100 | 2000 |
| 코드 생성 | 0.3 | 0.9 | 30 | 2048 |

In [ ]:
# 레시피 1: 사실 기반 Q&A
prompt = "파이썬으로 간단한 계산기를 만드는 방법을 설명해주세요."

text = generate(
    prompt,
    temperature=0.1,
    top_p=0.8,
    max_output_tokens=500,
)
print("[사실_기반_QA] temperature=0.1, top_p=0.8")
print(text[:300], "...")

In [ ]:
# 레시피 2: 자연스러운 대화
text = generate(
    prompt,
    temperature=0.8,
    top_p=0.95,
    max_output_tokens=800,
)
print("[자연스러운_대화] temperature=0.8, top_p=0.95")
print(text[:300], "...")

In [ ]:
# 레시피 3: 창의적 글쓰기
text = generate(
    prompt,
    temperature=1.4,
    top_p=0.99,
    max_output_tokens=1000,
)
print("[창의적_글쓰기] temperature=1.4, top_p=0.99")
print(text[:300], "...")

In [ ]:
# 레시피 4: 코드 생성
text = generate(
    prompt,
    temperature=0.3,
    top_p=0.9,
    max_output_tokens=1024,
    stop_sequences=["```\n\n"],
)
print("[코드_생성] temperature=0.3, top_p=0.9")
print(text[:300], "...")

---
## 9. 파라미터 튜닝 실험 — 감성 문장 생성

같은 프롬프트에 temperature와 top_p를 달리해 결과가 어떻게 달라지는지 비교합니다.

In [ ]:
# temperature=0.1, top_p=0.7 (매우 결정적)
prompt = "봄날 오후의 느낌을 한 문장으로 표현해주세요."

text = generate(prompt, temperature=0.1, top_p=0.7, max_output_tokens=80)
print(f"temp=0.1, top_p=0.7: {text.strip()}")

In [ ]:
# temperature=0.7, top_p=0.9 (균형)
text = generate(prompt, temperature=0.7, top_p=0.9, max_output_tokens=80)
print(f"temp=0.7, top_p=0.9: {text.strip()}")

In [ ]:
# temperature=1.2, top_p=0.95 (창의적)
text = generate(prompt, temperature=1.2, top_p=0.95, max_output_tokens=80)
print(f"temp=1.2, top_p=0.95: {text.strip()}")

In [ ]:
# temperature=1.8, top_p=0.99 (매우 창의적)
text = generate(prompt, temperature=1.8, top_p=0.99, max_output_tokens=80)
print(f"temp=1.8, top_p=0.99: {text.strip()}")

---
## 10. 요청 한도 오류와 안전한 실행

Gemini API 무료 티어에서는 짧은 시간에 너무 많은 요청을 보내면 다음 오류가 발생합니다:

```text
429 RESOURCE_EXHAUSTED
Quota exceeded for metric: generate_content_free_tier_requests
Please retry in 12s.
```

이 노트북은 셀을 **하나씩 실행**해 오류를 피합니다.  
429가 발생하면 셀 출력에 안내 메시지가 표시되니 잠시 기다린 뒤 재실행하세요.

### 429 오류 처리 패턴

```python
from google.genai import errors

try:
    response = client.models.generate_content(...)
except errors.APIError as error:
    if error.code == 429:
        print("요청 한도 초과: 잠시 기다린 뒤 다시 실행하세요.")
```

---
## 핵심 요약

| 파라미터 | 범위 | 높을수록 | 낮을수록 |
|----------|------|----------|----------|
| `temperature` | 0.0 ~ 2.0 | 창의적, 무작위 | 결정적, 일관성 |
| `top_p` | 0.0 ~ 1.0 | 다양한 어휘 사용 | 고확률 어휘만 사용 |
| `top_k` | 1 ~ | 다양성 증가 | 결정성 증가 |
| `max_output_tokens` | 1 ~ | 긴 답변 허용 | 짧은 답변 강제 |
